In [1]:
import pandas as pd 
import numpy as np

In [2]:
# 1. Load the Excel workbook
excel_file = 'Pharmacy_data.xlsx'
xls = pd.ExcelFile(excel_file)

# Display all sheet names found in the workbook
print("Sheets found:", xls.sheet_names)

# 2. Loop through every sheet and save as an individual CSV
for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    csv_name = f"{sheet}.csv"
    df.to_csv(csv_name, index=False)
    print(f"Successfully created: {csv_name} ({df.shape[0]} rows, {df.shape[1]} columns)")

Sheets found: ['FactSales', 'DimDate', 'DimPharmacy', 'DimProduct']
Successfully created: FactSales.csv (62139 rows, 9 columns)
Successfully created: DimDate.csv (731 rows, 7 columns)
Successfully created: DimPharmacy.csv (120 rows, 10 columns)
Successfully created: DimProduct.csv (220 rows, 11 columns)


In [4]:
# Load the CSV files created from the Excel extraction
fact_sales = pd.read_csv('FactSales.csv')
dim_date = pd.read_csv('DimDate.csv')
dim_pharmacy = pd.read_csv('DimPharmacy.csv')
dim_product = pd.read_csv('DimProduct.csv')

print("Data successfully loaded into memory!")
print(f"FactSales shape: {fact_sales.shape}")
print(f"DimDate shape: {dim_date.shape}")
print(f"DimPharmacy shape: {dim_pharmacy.shape}")
print(f"DimProduct shape: {dim_product.shape}")

Data successfully loaded into memory!
FactSales shape: (62139, 9)
DimDate shape: (731, 7)
DimPharmacy shape: (120, 10)
DimProduct shape: (220, 11)


In [5]:
# 1. Clean FactSales (Transactions)
fact_sales_clean = fact_sales.drop_duplicates().copy()

# Handle numeric anomalies (replace negative revenue or units with NaN)
numeric_cols = fact_sales_clean.select_dtypes(include=[np.number]).columns
for col in ['Revenue', 'Units', 'Cost', 'Profit']:
    if col in numeric_cols:
        fact_sales_clean.loc[fact_sales_clean[col] < 0, col] = np.nan

In [6]:
# 2. Clean DimProduct
dim_product_clean = dim_product.drop_duplicates().copy()
string_cols_prod = dim_product_clean.select_dtypes(include=['object']).columns
for col in string_cols_prod:
    dim_product_clean[col] = dim_product_clean[col].fillna('Unknown').str.strip()

In [7]:
# 3. Clean DimPharmacy
dim_pharmacy_clean = dim_pharmacy.drop_duplicates().copy()
string_cols_pharm = dim_pharmacy_clean.select_dtypes(include=['object']).columns
for col in string_cols_pharm:
    dim_pharmacy_clean[col] = dim_pharmacy_clean[col].fillna('Unknown').str.strip()

In [8]:
# 4. Clean DimDate
dim_date_clean = dim_date.drop_duplicates().copy()

print("Data Cleaning Complete:")
print(f"- FactSales null count after cleaning:\n{fact_sales_clean.isnull().sum()}")

Data Cleaning Complete:
- FactSales null count after cleaning:
SalesID       0
DateKey       0
PharmacyID    0
ProductID     0
UnitsSold     0
RevenueEUR    0
CostEUR       0
MarginEUR     0
PromoFlag     0
dtype: int64


In [15]:
# 1. Calculate Profit Margin Percentage
# Using your exact columns: 'MarginEUR' and 'RevenueEUR'
if 'RevenueEUR' in fact_sales_clean.columns and 'MarginEUR' in fact_sales_clean.columns:
    # Using np.where to avoid division by zero errors if Revenue is 0
    fact_sales_clean['Profit_Margin_Pct'] = np.where(
        fact_sales_clean['RevenueEUR'] > 0,
        (fact_sales_clean['MarginEUR'] / fact_sales_clean['RevenueEUR']) * 100,
        0
    ).round(2)
    print("Success: 'Profit_Margin_Pct' calculated.")

# 2. Calculate Average Selling Price (ASP)
# Using your exact columns: 'RevenueEUR' and 'UnitsSold'
if 'RevenueEUR' in fact_sales_clean.columns and 'UnitsSold' in fact_sales_clean.columns:
    fact_sales_clean['ASP'] = np.where(
        fact_sales_clean['UnitsSold'] > 0,
        (fact_sales_clean['RevenueEUR'] / fact_sales_clean['UnitsSold']),
        0
    ).round(2)
    print("Success: 'ASP' calculated.")

# 3. Bin Profit Margins into Strategic Tiers
def categorize_margin(margin):
    if pd.isna(margin):
        return 'Unknown'
    elif margin >= 30:
        return 'High Margin'
    elif margin >= 15:
        return 'Medium Margin'
    else:
        return 'Low Margin'

if 'Profit_Margin_Pct' in fact_sales_clean.columns:
    fact_sales_clean['Margin_Tier'] = fact_sales_clean['Profit_Margin_Pct'].apply(categorize_margin)
    print("Success: 'Margin_Tier' applied.")

# Verify the new columns
print("\nNew columns in FactSales:", fact_sales_clean[['Profit_Margin_Pct', 'ASP', 'Margin_Tier']].head(3))

Success: 'Profit_Margin_Pct' calculated.
Success: 'ASP' calculated.
Success: 'Margin_Tier' applied.

New columns in FactSales:    Profit_Margin_Pct    ASP  Margin_Tier
0              31.64  25.62  High Margin
1              33.86  12.97  High Margin
2              37.20  15.89  High Margin


In [16]:
# 4. Extract Calendar Features for DimDate
# Checking for either 'Date' or 'DateKey' based on common naming conventions
date_col = 'Date' if 'Date' in dim_date_clean.columns else 'DateKey' if 'DateKey' in dim_date_clean.columns else None

if date_col:
    # Convert to datetime format
    dim_date_clean[date_col] = pd.to_datetime(dim_date_clean[date_col])
    
    # Extract features
    dim_date_clean['Year'] = dim_date_clean[date_col].dt.year
    dim_date_clean['Quarter'] = 'Q' + dim_date_clean[date_col].dt.quarter.astype(str)
    dim_date_clean['Month_Name'] = dim_date_clean[date_col].dt.strftime('%b')
    dim_date_clean['Is_Weekend'] = dim_date_clean[date_col].dt.dayofweek.isin([5, 6]).astype(int)
    
    print(f"Success: Calendar features extracted from '{date_col}'.")
    print(dim_date_clean[['Year', 'Quarter', 'Month_Name', 'Is_Weekend']].head(3))
else:
    print("Error: Could not find the date column in DimDate.")

Success: Calendar features extracted from 'Date'.
   Year Quarter Month_Name  Is_Weekend
0  2024      Q1        Jan           0
1  2024      Q1        Jan           0
2  2024      Q1        Jan           0


In [ ]:
# Save the finalized, clean, and engineered dataframes to CSVs
fact_sales_clean.to_csv('FactSales_Final.csv', index=False)
dim_product_clean.to_csv('DimProduct_Final.csv', index=False)
dim_pharmacy_clean.to_csv('DimPharmacy_Final.csv', index=False)
dim_date_clean.to_csv('DimDate_Final.csv', index=False)

print("All final datasets exported successfully! Ready for the SQL phase.") 

All final datasets exported successfully! Ready for the SQL phase.
